In [12]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import torch.nn.functional as F

In [13]:
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [14]:
len(words)

32033

In [15]:
# building the vocabulary of the characters and mapping them to /from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [21]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next token
X, Y = [], []
for w in words[:5]:
    
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} ----> {itos[ix]}")
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [22]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [23]:
C = torch.randn((27,2))

In [28]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [29]:
W1 = torch.randn(6,100)
b1 = torch.randn(100)

In [30]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1)

In [31]:
h

tensor([[-0.9917, -0.6537, -0.7675,  ...,  0.8778, -0.9713, -0.9991],
        [-0.6476,  0.9991, -0.9441,  ..., -0.7219, -0.9489, -0.9975],
        [ 0.9219,  1.0000, -0.9995,  ..., -0.8139, -0.5039, -0.9974],
        ...,
        [ 0.9991,  0.7462,  0.9635,  ..., -1.0000,  0.9994, -0.8064],
        [ 0.8714, -1.0000,  1.0000,  ...,  0.9887,  0.0526, -0.3340],
        [ 0.9618,  1.0000,  0.0204,  ..., -0.9746,  0.6714, -0.9609]])

In [32]:
h.shape

torch.Size([32, 100])

In [33]:
W2 = torch.randn(100,27)
b2 = torch.randn(27)

In [34]:
logits = h @ W2 + b2

In [35]:
counts = logits.exp()

In [36]:
prob = counts / counts.sum(1,keepdims=True)

In [37]:
loss = -prob[torch.arange(32),Y].log().mean()
loss

tensor(13.8094)

In [39]:
loss.item()

13.809406280517578

In [40]:
# summarizing earlier code

In [42]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [43]:
g = torch.Generator().manual_seed(2147483647) # for reproducability
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn((27), generator=g)
parameters = [C,W1,b1,W2,b2]

In [44]:
sum(p.nelement() for p in parameters) # total number of parameters

3481

In [45]:
emb = C[X]
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32,100)
logits = h @ W2 + b2 # (32,27)
counts = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)